In [1]:
import torch, transformers
import json

model_8B_id = "meta-llama/Meta-Llama-3-8B-Instruct"
model_3B_id = "meta-llama/Llama-3.2-3B-Instruct"
tok = transformers.AutoTokenizer.from_pretrained(model_8B_id)
model = transformers.AutoModelForCausalLM.from_pretrained(
    model_8B_id, torch_dtype=torch.float16, device_map="cuda"
)

d:\GeoTKG\llama_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████| 4/4 [00:28<00:00,  7.08s/it]


In [115]:
def get_test_data():
    examples = []
    with open("C:\\Users\\hazza\\OneDrive\\Desktop\\GeoTKG\\cleandata\\tie\\test.json", "r") as f:
        examples=[json.loads(line) for line in f]
    return examples

def get_ner_prompt(text, method):
    if method == "event and time":
        tags = "EVENT, DATE, TIME, DURATION, SET"
    elif method == "geoscience":
        tags = "LOCATION, MINERAL, ORE_DEPOSIT, ROCK, STRAT, TIMESCALE"
    sys = f'''You are a Named Entity Recognition (NER) system for tagging {method} entities. Identify and classify entities in the text based on the entity types: {tags}. 
            Each entity should be represented as a tuple (entity surface text, type) in valid JSON format.
            Return ONLY valid JSON (no markdown, no commentary). Escape double quotes as \".'''
    
    user = f'''Extract entities from the following text: {text}'''
    messages = [
        {"role": "system", "content": sys},
        {"role": "user", "content": user}
    ]
    return messages

def get_norm_prompt(text, dct):
    sys = '''
        You are a geological timescale and calendar time normalization system. Normalize the time expressions that have been tagged in the text using the document creation time as anchor for calendar time.
        Each time mention should be represented as a tuple: (surface text, normalized value).
        Geological timescale normalized values should be in ma (million years ago).
        Calendar time expressions should be in ISO 8601 format (YYYY-MM-DD).
        Return ONLY valid JSON (no markdown, no commentary). Escape double quotes as \".
    '''
    user = f'Normalize time expressions from the following passage which has a document creation time of {dct}: {text}'
    messages = [
    {"role":"system","content":sys},
    {"role":"user","content":user}
    ]
    return messages

def get_tkg_prompt(text, dct):
    text = " ".join([wrd for sent in text for wrd in sent])
    sys = '''
        You are an information extraction system for geoscience and general texts.
        Extract (1) times, (2) quintuples (events), and (3) temporal-relation triples.
        Return ONLY valid JSON (no markdown, no commentary). Escape double quotes as \".

        Gregorian Calendar Times and Geological Timescales
        - Each time has format: ["T#", "surface text", "normalized value or null", "DATE|DURATION|SET|TIME|GEO_TIME"]
        - Reuse the same T# if the surface text repeats.

        Quintuples (events)
        - Each event is one quintuple: ["E#", "subject or null", "event string", "object or null", "T# or null", "T# or null"]
        - Event string must be short (just the trigger words).
        - E# assigned in order of first mention; reuse IDs for duplicates.

        Temporal triples
        - BEFORE: event1 ends before event2 starts
        - AFTER:  event1 starts after event2 ends
        - DURING: event1 occurs fully within event2
        - CONTAINS: event1 fully contains event2
        - IDENTITY/EQUALS: same event/time span
        - OVERLAPS: partial intersection
        - Each relation is ["E#", "BEFORE|AFTER|DURING|CONTAINS|IDENTITY|EQUALS|OVERLAPS", "E#"]
        - Only E# allowed, never T#.
        - Each unordered event pair appears at most once.

        Validation
        - IDs sequential by first mention (E1, E2 ...; T1, T2 ...).
        - All T# in quintuples must exist in times.
        - JSON must be valid: no trailing commas, no comments.
    '''
    user = f'Extract events and temporal relations from the following passage (document creation time: {dct}):    {text}'
    messages = [
    {"role":"system","content":sys},
    {"role":"user","content":user}
    ]
    return messages

def jsonify(output):
    try:
        return json.loads(output)
    except json.JSONDecodeError:
        try:
            return json.loads(output+"}")
        except json.JSONDecodeError:
            return None

def inference(model, tok, prompt, max_new_tokens=4000):
    input_prompt = tok.apply_chat_template(prompt, add_generation_prompt=True, tokenize=False)
    inputs = tok(input_prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, temperature=0.2, do_sample=False)
    gen_tokens = out[0, inputs.input_ids.shape[-1]:]
    decodings = tok.decode(gen_tokens, skip_special_tokens=True)
    prediction = decodings.strip(":\n`").lstrip("```json\n").rstrip("\n```")
    return prediction

def norm_preprocess(text, times):
    out = ""
    for sn, sent in enumerate(text):
        sent_times = [t['offset'] for t in times if t['sent_id'] == sn]
        sent_times = sorted(sent_times, key=lambda x: x[0], reverse=True)

        for st, en in sent_times:
            sent.insert(en,"</timex>")
            sent.insert(st,"<timex>")
        out += " ".join(sent) + " "
    return out.strip()

def post_processing(preds):
    out = {}
    for fn, pred in enumerate(preds):
        json_out = jsonify(pred['pred'])
        if json_out is None:
            testy = pred['pred'].partition('{')[-1].rpartition('}')[0]
            json_out = jsonify('{'+testy+'}')
        
        if json_out is None or json_out == {}:
            testy = pred['pred'].partition('{')[-1][:-6]
            json_out = jsonify('{'+testy+'}')

        if json_out is None:
            testy = pred['pred'].partition('{')[-1][:-1]
            json_out = jsonify('{'+testy+'}')

        if json_out is None:
            testy = pred['pred'].partition('{')[-1].rpartition(']')[0]
            json_out = jsonify('{'+testy+'}')

        if type(json_out) is list:
            json_out = json_out[0]

        keys = list(json_out.keys())
        keys.remove('times')
        keys.remove('quintuples')
        json_out['triples'] = json_out.pop(keys[0])

        out[fn] = {'text':pred['text'], 'pred':json_out}

def ner_post_processing(ner_preds):
    poster = []

    for fn, pred in enumerate(ner_preds):
        if fn==121:
            continue
        stripped = pred['pred'].replace('{"entity surface text": ', '[').replace('"type": ', '').replace("}",']').replace('{"entity": ',"[").replace("{","[")
        texty = stripped.partition('[')[-1].rpartition(']')[0]
        pred_json = jsonify("["+texty+"]")
        if pred_json is None:
            print(fn)
            pred_json = texty
        poster.append({'text':pred['text'], 'pred':pred_json})
    return poster


In [116]:
test_data = get_test_data()

In [ ]:
prediction_type = "ner"

if prediction_type == "tkg":
    chat_tkg_preds = []
    file_num = 1
    for example in test_data:
        dct = [inst['value'] for inst in example['instances'] if inst['type'] != "EVENT" and inst['id'] == 0][0]
        prompt = get_tkg_prompt(example['text'], dct)
        prediction = inference(model, tok, prompt)
        chat_tkg_preds.append({"text":example['text'], "pred":prediction})
        print(f"Processed example {file_num} / {len(test_data)}")
        file_num += 1
elif prediction_type == "ner":
    print("----- RUNNING NER PREDICTIONS -----")
    chat_ner_preds = []
    out_file_name = "llama3.2-8B-ner-preds.json"
    file_num = 1
    for example in test_data[121:122]:
        prompt = get_ner_prompt(example['text'], "event and time")
        prediction = inference(model, tok, prompt, max_new_tokens=1000)
        chat_ner_preds.append({"text":example['text'], "pred":prediction})
        print(f"Processed example {file_num} / {len(test_data)}")
        file_num += 1
elif prediction_type == "norm":
    chat_norm_preds = []
    file_num = 1
    for example in test_data:
        dct = [inst['value'] for inst in example['instances'] if inst['type'] != "EVENT" and inst['id'] == 0][0]
        prompt = get_norm_prompt(norm_preprocess(example['text'], [instance for instance in example['instances'] if instance['type'] != "EVENT" and instance['id'] != 0]), dct)
        prediction = inference(model, tok, prompt, max_new_tokens=1000)
        chat_norm_preds.append({"text":example['text'], "pred":prediction})
        print(f"Processed example {file_num} / {len(test_data)}")
        file_num += 1

In [ ]:
failed_times = []
failed_events = []
failed_trips = []
for fn, pred in enumerate(out):
    if out[pred]['pred'] is None:
        failed_times.append(fn)
        failed_events.append(fn)
        failed_trips.append(fn)
    else:
        try:
            x = out[pred]['pred']['times']
        except KeyError:
            failed_times.append(fn)

        try:
            y = out[pred]['pred']['quintuples']
        except KeyError:
            failed_events.append(fn)

        try:
            z = out[pred]['pred']['triples']
        except KeyError:
            failed_trips.append(fn)

failed_times, failed_events, failed_trips

In [130]:
import json
with open("idk2.json", 'w') as json_file:
    for sample in poster:
        json_file.write(json.dumps(sample)+"\n")

In [108]:
import json
with open("llama3.2-8B-ner-preds.json", "r") as f:
    examples=[json.loads(line) for line in f]

In [107]:
poster[140]['pred']
#json.loads(poster[140]['pred'])

'\n    ["Before the arrival of Keep, which was launched this week, there was no default note-taking app for Android.", "EVENT"],\n  ["It was a glaring hole, considering that Apple\'s iPhone has built-in Notes and Reminders apps that can be powered by Siri.","EVENT"],\n    ["Instead of settling for a bare bones app to fill the void, the search giant took things one step further.", "EVENT"],\n    ["Keep is not simply just a place to bank whatever random half-thoughts come to mind : Users can construct to-do lists, stash photos, and color code your notes -- all in one well-designed and easy-to-use interface.", "EVENT"],\n    ["It\'s easy to foresee the day the when users will be able to send anything from their Web browser or Maps directly to Keep.", "EVENT"],\n  ["The prospect of Keep incorporating features of services such as Pinterest or Pocket, or even making it easy to catalog streaming media, could turn it into something big.", "EVENT"],\n    ["That should scare Evernote.","EVENT"],

In [109]:
poster = ner_post_processing(examples)

105


In [128]:
for fn, sample in enumerate(poster):
    try:
        print(fn, sample['pred'][0])
    except IndexError:
        print(fn, type(sample))

0 ['Friday', 'DATE']
1 ['NAIROBI', 'LOCATION']
2 ['DAR es Salaam', 'LOCATION']
3 ['Monday', 'DATE']
4 ['MANILA', 'LOCATION']
5 ['UNITED NATIONS', 'EVENT']
6 ['NAIROBI', 'LOCATION']
7 ['AP', 'EVENT']
8 ['Wednesday', 'DATE']
9 ['DAR', 'LOCATION']
10 ['NAIROBI', 'LOCATION']
11 ['WARSAW', 'EVENT']
12 ['Preliminary DNA tests', 'EVENT']
13 ['Washington', 'LOCATION']
14 ['INDEPENDENCE', 'EVENT']
15 ['Dr. Barnett Slepian', 'EVENT']
16 ['Thursday', 'DATE']
17 ['Slepian', 'PERSON']
18 ['Osama Bin Laden', 'EVENT']
19 ['Aug. 7, 1998', 'DATE']
20 ['last year', 'DATE']
21 ['Sunday', 'DATE']
22 ['DAR es Salaam', 'LOCATION']
23 ['Elian Gonzalez', 'EVENT']
24 ['Elian Gonzalez', 'EVENT']
25 ['MIAMI', 'LOCATION']
26 ['HAVANA', 'LOCATION']
27 ['Elian Gonzalez', 'EVENT']
28 ['Monday', 'DATE']
29 ['HAVANA', 'LOCATION']
30 ['Elian Gonzalez', 'EVENT']
31 ["Elian Gonzalez's father", 'EVENT']
32 ['HAVANA', 'LOCATION']
33 ['HAVANA', 'LOCATION']
34 ['Elian Gonzalez', 'EVENT']
35 ['HAVANA', 'LOCATION']
36 ['HAVANA

In [110]:
help105 = "[\n    {\"entity surface text\": \"Saturday\", \"type\": \"DATE\"},\n    {\"entity surface text\": \"April 25\", \"type\": \"DATE\"},\n    {\"entity surface text\": \"two years ago\", \"type\": \"DURATION\"},\n    {\"entity surface text\": \"this week\", \"type\": \"DURATION\"},\n    {\"entity surface text\": \"Saturday\", \"type\": \"DATE\"},\n    {\"entity surface text\": \"April 25\", \"type\": \"DATE\"},\n    {\"entity surface text\": \"this week\", \"type\": \"DURATION\"},\n    {\"entity surface text\": \"Saturday\", \"type\": \"DATE\"},\n    {\"entity surface text\": \"April 25\", \"type\": \"DATE\"},\n    {\"entity surface text\": \"this week\", \"type\": \"DURATION\"},\n    {\"entity surface text\": \"Saturday\", \"type\": \"DATE\"},\n    {\"entity surface text\": \"April 25\", \"type\": \"DATE\"},\n    {\"entity surface text\": \"this week\", \"type\": \"DURATION\"},\n    {\"entity surface text\": \"Saturday\", \"type\": \"DATE\"},\n    {\"entity surface text\": \"April 25\", \"type\": \"DATE\"},\n    {\"entity surface text\": \"this week\", \"type\": \"DURATION\"},\n    {\"entity surface text\": \"Saturday\", \"type\": \"DATE\"},\n    {\"entity surface text\": \"April 25\", \"type\": \"DATE\"},\n    {\"entity surface text\": \"this week\", \"type\": \"DURATION\"},\n    {\"entity surface text\": \"Saturday\", \"type\": \"DATE\"},\n    {\"entity surface text\": \"April 25\", \"type\": \"DATE\"},\n    {\"entity surface text\": \"this week\", \"type\": \"DURATION\"},\n    {\"entity surface text\": \"Saturday\", \"type\": \"DATE\"},\n    {\"entity surface text\": \"April 25\", \"type\": \"DATE\"},\n    {\"entity surface text\": \"this week\", \"type\": \"DURATION\"},\n    {\"entity surface text\": \"Saturday\", \"type\": \"DATE\"},\n    {\"entity surface text\": \"April 25\", \"type\": \"DATE\"},\n    {\"entity surface text\": \"this week\", \"type\": \"DURATION\"},\n    {\"entity surface text\": \"Saturday\", \"type\": \"DATE\"},\n    {\"entity surface text\": \"April 25\", \"type\": \"DATE\"},\n    {\"entity surface text\": \"this week\", \"type\": \"DURATION\"},\n    {\"entity surface text\": \"Saturday\", \"type\": \"DATE\"},\n    {\"entity surface text\": \"April 25\", \"type\": \"DATE\"},\n    {\"entity surface text\": \"this week\", \"type\": \"DURATION\"},\n    {\"entity surface text\": \"Saturday\", \"type\": \"DATE\"},\n    {\"entity surface text\": \"April 25\", \"type\": \"DATE\"},\n    {\"entity surface text\": \"this week\", \"type\": \"DURATION\"},\n    {\"entity surface text\": \"Saturday\", \"type\": \"DATE\"},\n    {\"entity surface text\": \"April 25\", \"type\": \"DATE\"},\n    {\"entity surface text\": \"this week\", \"type\": \"DURATION\"},\n    {\"entity surface text\": \"Saturday\", \"type\": \"DATE\"},\n    {\"entity surface text\": \"April 25\", \"type\": \"DATE\"},\n    {\"entity surface text\": \"this week\", \"type\": \"DURATION\"},\n    {\"entity surface text\": \"Saturday\", \"type\": \"DATE\"},\n    {\"entity surface text\": \"April 25\", \"type\": \"DATE\"},\n    {\"entity surface text\": \"this week\", \"type\": \"DURATION\"},\n    {\"entity surface text\": \"Saturday\", \"type\": \"DATE\"},\n    {\"entity surface text\": \"April 25\", \"type\": \"DATE\"},\n    {\"entity surface text\": \"this week\", \"type\": \"DURATION\"},\n    {\"entity surface text\": \"Saturday\", \"type\": \"DATE\"},\n    {\"entity surface text\": \"April 25\", \"type\": \"DATE\"},\n    {\"entity surface text\": \"this week\", \"type\": \"DURATION\"},\n    {\"entity surface text\": \"Saturday\", \"type\": \"DATE\"},\n    {\"entity surface text\": \"April 25\", \"type\": \"DATE\"},\n    {\"entity surface text\": \"this week\", \"type\": \"DURATION\"},\n    {\"entity surface text\": \"Saturday\", \"type\": \"DATE\"},\n    {\"entity surface text\": \"April 25\", \"type\": \"DATE\"},\n    {\"entity surface text\": \"this week\", \"type\": \"DURATION\"},\n    {\"entity surface text\": \"Saturday\", \"type\": \"DATE\"},\n    {\"entity surface text\": \"April 25\", \"type\": \"DATE\"},\n    {\"entity surface text\": \"this week\", \"type\": \"DURATION\"}\n   }"
help105 = help105.replace('{"entity surface text": ', '[').replace('"type": ', '').replace("}",']')
poster[105]['pred'] = json.loads(help105)

In [117]:
len(test_data)

151

In [118]:
len(poster)

150

In [123]:
examine = [] 

for exa in test_data:
    iii = []
    for sent in exa['text']: 
        for wrd in sent:
            iii.append(wrd)
    examine.append(" ".join(iii))

In [127]:
hhh = []
for exa in poster:
    iii = []
    for sent in exa['text']:
        for wrd in sent:
            iii.append(wrd)
    hhh.append(" ".join(iii))

for fn, j in enumerate(examine):
    if j not in hhh:
        print(fn, j)

121 " So far , the offensive   is progressing with dramatic success , " said   a buoyant Gen. Norman Schwarzkopf , commander of U.S. forces . Similarly , while cautioning about the uncertainty of early battle reports , White House spokesman Marlin Fitzwater said   late yesterday   that " the operation   has been very successful . " Amid reports that thousands of Iraqi soldiers had surrendered , administration aides were also upbeat in private , with one even talking   of victory within a week . But even continued military success carries   political and diplomatic risks for President Bush and the U.S. The allied rejection   of the last - minute Soviet - led diplomatic effort   to avoid   the ground war   enabled   Mr. Bush to seize   the initiative from an Iraq seemingly bent on dictating   peace terms . But it has offended   some , especially in Arab countries , who now believe   that Mr. Bush 's real objectives are the demise   of Saddam Hussein and the destruction   of the Iraqi mil

In [125]:
poster[0]['text']

[['NAIROBI',
  ',',
  'Kenya',
  '(',
  'AP',
  ')',
  '_',
  'Suspected',
  'bombs',
  'exploded',
  ' ',
  'outside',
  'the',
  'U.S.',
  'embassies',
  'in',
  'the',
  'Kenyan',
  'and',
  'Tanzanian',
  'capitals',
  'Friday',
  ',',
  'killing',
  ' ',
  'dozens',
  'of',
  'people',
  ',',
  'witnesses',
  'said',
  '.The',
  'American',
  'ambassador',
  'to',
  'Kenya',
  'was',
  'among',
  'hundreds',
  'injured',
  ',',
  'a',
  'local',
  'TV',
  'said',
  '.``It',
  'was',
  'definitely',
  'a',
  'bomb',
  ',',
  "''",
  'said',
  ' ',
  'a',
  'U.S.',
  'Embassy',
  'official',
  'in',
  'Nairobi',
  ',',
  'who',
  'refused',
  ' ',
  'to',
  'identify',
  ' ',
  'himself',
  '.'],
 ['`',
  '`',
  'You',
  'can',
  'see',
  ' ',
  'a',
  'huge',
  'crater',
  'behind',
  'the',
  'building',
  ',',
  'and',
  'a',
  'bomb',
  'went',
  ' ',
  'off',
  'at',
  'the',
  'embassy',
  'in',
  'Tanzania',
  'at',
  'the',
  'same',
  'time',
  ',',
  "''",
  'he',
  'said'